In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
CALENDAR_DIR = CONFIG_DIR / "calendars"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Task 5 input/output paths
TRAIN_PATH = OBJ2_DATA_DIR / "demand_model_train_ready.parquet"
WEATHER_FUTURE_PATH = OBJ2_DATA_DIR / "weather_future_features_daily.parquet"
MEMBER_SELECTION_PATH = OBJ2_DATA_DIR / "ukcp18_member_selection.csv"

HOLIDAY_PATH = CALENDAR_DIR / "holiday_calendar_2010_2045.csv"
if not HOLIDAY_PATH.exists():
    HOLIDAY_PATH = PREPROCESSING_DATA_DIR / "holiday_calendar_2010_2045.csv"

MODEL_PATH = OBJ2_MODEL_DIR / "demand_model_xgb_model3.pkl"

FUTURE_MODEL_INPUT_PATH = OBJ2_DATA_DIR / "future_demand_model_input.parquet"
FUTURE_DAILY_CLIMATE_ONLY_PATH = OBJ2_DATA_DIR / "future_daily_demand_climate_only.parquet"
FUTURE_DAILY_CLIMATE_ONLY_README_PATH = OBJ2_VALIDATION_DIR / "future_daily_demand_climate_only_README.md"


In [ ]:
member_sel_df = pd.read_csv(MEMBER_SELECTION_PATH)

selected_members = (
    member_sel_df
    .loc[member_sel_df["selected_flag"].astype(int).eq(1), "climate_member"]
    .astype(str)
    .unique()
    .tolist()
)

print("Selected climate members:", selected_members)

In [ ]:
weather_future_df = pd.read_parquet(WEATHER_FUTURE_PATH).copy()
weather_future_df["date"] = pd.to_datetime(weather_future_df["date"])
weather_future_df["climate_member"] = weather_future_df["climate_member"].astype(str)

member_sel_df = pd.read_csv(MEMBER_SELECTION_PATH)

selected_members = (
    member_sel_df
    .loc[member_sel_df["selected_flag"].astype(int).eq(1), "climate_member"]
    .astype(str)
    .tolist()
)

future_start = pd.Timestamp("2030-01-01")
future_end = pd.Timestamp("2045-12-31")

weather_future_df = weather_future_df[
    weather_future_df["climate_member"].isin(selected_members)
    & weather_future_df["date"].between(future_start, future_end)
].copy()

print(weather_future_df["date"].min())
print(weather_future_df["date"].max())
print(weather_future_df.shape)
print(weather_future_df["climate_member"].unique())

In [ ]:
# Calendar features
weather_future_df["year"] = weather_future_df["date"].dt.year.astype(int)
weather_future_df["month"] = weather_future_df["date"].dt.month.astype(int)
weather_future_df["day_of_week"] = weather_future_df["date"].dt.dayofweek.astype(int)
weather_future_df["day_of_year"] = weather_future_df["date"].dt.dayofyear.astype(int)
weather_future_df["is_weekend"] = weather_future_df["day_of_week"].isin([5, 6]).astype(int)

origin_date = pd.Timestamp("2010-01-01")
weather_future_df["time_index"] = (
    weather_future_df["date"] - origin_date
).dt.days.astype(int)

holiday_df = pd.read_csv(HOLIDAY_PATH)
holiday_df["date"] = pd.to_datetime(holiday_df["date"])

future_input = weather_future_df.merge(holiday_df, on="date", how="left")

In [ ]:
required_future_input_cols = [
    "date",
    "year",
    "month",
    "day_of_week",
    "day_of_year",
    "time_index",
    "is_weekend",
    "is_holiday_eng_wales",
    "is_holiday_scotland",
    "is_holiday_gb_any",
    "climate_member",
    "tasmin_gb_c",
    "tasmax_gb_c",
    "tmean_gb_c",
    "hdd",
    "cdd",
]

missing_cols = sorted(set(required_future_input_cols) - set(future_input.columns))
assert not missing_cols, f"Missing required columns: {missing_cols}"

assert future_input["date"].min() == pd.Timestamp("2030-01-01")
assert future_input["date"].max() == pd.Timestamp("2045-12-31")

assert future_input.duplicated(["date", "climate_member"]).sum() == 0

assert future_input[
    ["is_holiday_eng_wales", "is_holiday_scotland", "is_holiday_gb_any"]
].isna().sum().sum() == 0

expected_days = len(pd.date_range("2030-01-01", "2045-12-31", freq="D"))
expected_rows = expected_days * len(selected_members)

print("Expected rows:", expected_rows)
print("Actual rows:", len(future_input))

assert len(future_input) == expected_rows

In [ ]:
null_mask = future_input.isna().any(axis=1)
future_input_null = future_input.loc[null_mask]

if future_input_null.empty:
    print("No null rows found.")
else:
    print(
        f"Date range for nulls: "
        f"{future_input_null['date'].min()} to {future_input_null['date'].max()}"
    )

In [ ]:
from xgboost import XGBRegressor

train_df = pd.read_parquet(TRAIN_PATH).copy()
train_df["date"] = pd.to_datetime(train_df["date"])
train_df = train_df.sort_values("date")

FEATURE_COLS = [
    "time_index",
    "day_of_year",
    "month",
    "day_of_week",
    "is_holiday_gb_any",
    "hdd",
    "cdd",
]

TARGET_COL = "nd_daily_mwh"

missing_train_cols = sorted(set(FEATURE_COLS + [TARGET_COL]) - set(train_df.columns))
assert not missing_train_cols, f"Training data missing columns: {missing_train_cols}"

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]

# Replace these with your actual selected/tuned Task 3 hyperparameters if you have them.
model = XGBRegressor(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

joblib.dump(model, MODEL_PATH)


In [ ]:
X_future = future_input[FEATURE_COLS].copy()

future_input["climate_only_daily_demand_mwh"] = model.predict(X_future)

assert (future_input["climate_only_daily_demand_mwh"] > 0).all(), \
    "Some climate-only demand predictions are non-positive."

future_model_input_cols = [
    "date",
    "year",
    "month",
    "day_of_week",
    "day_of_year",
    "time_index",
    "is_weekend",
    "is_holiday_eng_wales",
    "is_holiday_scotland",
    "is_holiday_gb_any",
    "climate_member",
    "tmean_gb_c",
    "hdd",
    "cdd",
]

future_output_cols = [
    "date",
    "year",
    "climate_member",
    "climate_only_daily_demand_mwh",
    "month",
    "day_of_week",
    "is_weekend",
    "is_holiday_gb_any",
    "tmean_gb_c",
    "hdd",
    "cdd",
]

future_input[future_model_input_cols].to_parquet(
    FUTURE_MODEL_INPUT_PATH,
    index=False,
)

future_daily_climate_only = future_input[future_output_cols].copy()

future_daily_climate_only.to_parquet(
    FUTURE_DAILY_CLIMATE_ONLY_PATH,
    index=False,
)


In [ ]:
readme_text = """# future_daily_demand_climate_only_README.md

Purpose:
Generate climate-only future daily GB National Demand before FES annual anchoring.

Inputs:
- weather_future_features_daily.parquet
- ukcp18_member_selection.csv
- holiday_calendar_2010_2045.csv
- demand_model_train_ready.parquet or saved demand_model_xgb_model3.pkl

Model:
Final Task 3 XGBoost Model 3 feature set:
time_index, day_of_year, month, day_of_week, is_holiday_gb_any, hdd, cdd.

Notes:
- Only selected UKCP18 climate members are used.
- Outputs are daily and use date as the key.
- Demand is in MWh.
- Climate-only demand is not the final future demand level; Task 6 anchors annual totals to FES ED1.
"""

(FUTURE_DAILY_CLIMATE_ONLY_README_PATH).write_text(readme_text)
